# Basic

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import polars as pl
import pickle

import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.graph_fct as graph_fct


In [ ]:
def build_combined_subgraphs_and_id2label():
    df_relations = pl.read_parquet(f"{config.BasicConfig().relation_path}")
    df_mapped = pl.read_parquet(f"{config.BasicConfig().mapped_path}")
    mapped_ids = df_mapped["id"].to_list()

    # remove root concept
    df_relations = df_relations.filter(~pl.col("dst.id").is_in(config.TokenizerParam().exclude_cpt))

    # build graph and combine
    whole_graph = graph_fct.build_relations_graph(df_relations, col_src="src.id", col_dst="dst.id", col_relation="relation")
    combined_subgraphs = graph_fct.get_combined_subgraphs_from_nodes(whole_graph, mapped_ids)
    df_cpt = pl.read_parquet(config.BasicConfig().concept_path).select("id", "label")
    id_to_label = dict(zip(df_cpt["id"], df_cpt["label"]))
    
    with open(config.ProcessedGraph().id_to_label, "wb") as f:
        pickle.dump(id_to_label, f)

    with open(config.ProcessedGraph().combined_subgraphs, "wb") as f:
        pickle.dump(combined_subgraphs, f)

def get_combined_combined_subgraphs_and_id2label():
    with open(config.ProcessedGraph().id_to_label, "rb") as f:
        id_to_label = pickle.load(f)

    with open(config.ProcessedGraph().combined_subgraphs, "rb") as f:
        combined_subgraphs = pickle.load(f)

    return id_to_label, combined_subgraphs

In [ ]:
graph_fct.build_combined_subgraphs_and_id2label()


In [4]:
id_to_label, combined_subgraphs = graph_fct.get_combined_combined_subgraphs_and_id2label()

In [6]:
import networkx as nx
nx.dag_longest_path_length(combined_subgraphs)   # returns the length (number of edges) of the longest path


30